## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. It also installs `scikit-learn`, which this analysis needs, into the same session. You can read every line of the code below before you run it.

# Seven European warehouses: rebuilding a map from distances

**Multidimensional Scaling (Matrix Format)** · a worked example from the StatNex teaching guide

**The story.** A logistics planner has only a table of straight-line distances (in hundreds of km) between seven European cities where the company has warehouses. Can a map be rebuilt from distances alone?

**The question.** Does metric MDS reproduce the real map of the warehouse network?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `scikit-learn`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "scikit-learn>=1.8"

## The table

This cell saves the table `09_multivariate/multidimensional_scaling_matrix.csv` (7 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
City,London,Paris,Brussels,Amsterdam,Frankfurt,Zurich,Berlin
London,0.0,3.43,3.21,3.57,6.38,7.76,9.32
Paris,3.43,0.0,2.64,4.3,4.78,4.88,8.77
Brussels,3.21,2.64,0.0,1.73,3.17,4.92,6.51
Amsterdam,3.57,4.3,1.73,0.0,3.64,6.13,5.77
Frankfurt,6.38,4.78,3.17,3.64,0.0,3.04,4.24
Zurich,7.76,4.88,4.92,6.13,3.04,0.0,6.69
Berlin,9.32,8.77,6.51,5.77,4.24,6.69,0.0
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.manifold import ClassicalMDS

pd.set_option("display.width", 120)

## The matrix

In [ ]:
# A square table: one row and one column per object. Only one triangle need be
# typed; when the other is blank it is filled in by symmetry, and a blank diagonal
# is 0.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["London", "Paris", "Brussels", "Amsterdam", "Frankfurt", "Zurich", "Berlin"]
label_column = ["City"]
matrix = data[columns].apply(pd.to_numeric, errors="coerce").to_numpy(float)
labels = data[label_column[0]].astype(str).tolist() if label_column else columns
n = len(matrix)
upper_blank = np.isnan(matrix[np.triu_indices(n, 1)]).all()
lower_blank = np.isnan(matrix[np.tril_indices(n, -1)]).all()
matrix = np.nan_to_num(matrix, nan=0.0)
if upper_blank or lower_blank:
    matrix = matrix + matrix.T - np.diag(np.diag(matrix))
dimensions = 2
print(f"{n} objects, {dimensions} dimension(s)")

# The entries are dissimilarities (distances).
dissimilarity = matrix

## Squared dissimilarities and the double-centred matrix (Torgerson)

In [ ]:
# B = -1/2 J D^2 J, with J the centring matrix. If the distances come from points,
# B holds the inner products of the centred points, so its eigenvectors are the map.
# scikit-learn's ClassicalMDS does this and the next step inside one call (below);
# they are written out here to print B and every eigenvalue, as the app does.
squared = dissimilarity**2
centring = np.eye(n) - np.ones((n, n)) / n
b = -0.5 * centring @ squared @ centring
print()
print(pd.DataFrame(squared, index=labels, columns=labels).round(4))
print()
print(pd.DataFrame(b, index=labels, columns=labels).round(4))

## Eigenvalues and Mardia's criteria

In [ ]:
# One eigenvalue per dimension, largest first. They add up to the total spread of the
# objects (the squared dissimilarities summed over every pair, divided by n), so each
# one is the part of that spread its dimension carries. Read Mardia's criteria down
# the column: the value at dimension 2 is the share a two-dimensional map keeps;
# near 1, the map loses little.
#
# A negative eigenvalue means the table cannot be drawn exactly in any number of
# dimensions: measured or rounded distances, or distances along a curved surface,
# are seldom exactly those of points on a flat map.
#
# Mardia's criteria (Mardia 1978) divide the running total of the positive
# eigenvalues by the size of *all* of them: the first by the sum of their absolute
# values, the second by the sum of their squares. A negative eigenvalue is part of
# the distances no map can show, so it counts in the total. These are the first
# number of R's cmdscale() GOF and Stata's -mdsmat- "Mardia fit measure 1 and 2".
# Against an eigenvalue that is not positive there is no dimension to keep, and the
# app prints "-" (NaN here).
eigenvalues, vectors = np.linalg.eigh((b + b.T) / 2)
order = np.argsort(-eigenvalues)
eigenvalues, vectors = eigenvalues[order], vectors[:, order]
# An eigenvalue no larger in size than 1e-10 times the largest is zero up to
# rounding: the solver returns zeros as noise of either sign, around 1e-15 of the
# largest. It is set to exactly 0, as the app does.
eigenvalues[np.abs(eigenvalues) <= 1e-10 * np.abs(eigenvalues).max()] = 0.0
kept = np.clip(eigenvalues, 0.0, None)
mardia = pd.DataFrame(
    {"eigenvalue": eigenvalues, "Mardia 1": kept.cumsum() / np.abs(eigenvalues).sum(),
     "Mardia 2": (kept**2).cumsum() / (eigenvalues**2).sum()},
    index=[f"Dim {j + 1}" for j in range(n)],
)
mardia.loc[eigenvalues <= 0, ["Mardia 1", "Mardia 2"]] = np.nan
print()
print(mardia)

# An axis and its mirror image are the same map, so each eigenvector is turned to make
# its first element positive. That choice also decides whether the map comes out
# mirrored against a real one: flip one axis (multiply it by -1) to undo it.
vectors = vectors * np.where(vectors[0] < 0, -1.0, 1.0)
print()
print(pd.DataFrame(vectors, index=labels, columns=mardia.index).round(4))

## Coordinates

In [ ]:
# Each kept eigenvector times the square root of its eigenvalue: scikit-learn's
# ClassicalMDS, given the dissimilarity table ("precomputed"), returns exactly these.
# It picks each axis's sign by its own rule, so the axes are turned to start
# positive as above. Objects close on the map are close in the table; the axes have
# no units of their own and are named by the objects at their two ends.
classical = ClassicalMDS(n_components=dimensions, metric="precomputed").fit(dissimilarity)
coordinates = classical.embedding_
coordinates = coordinates * np.where(coordinates[0] < 0, -1.0, 1.0)
dims = [f"Dim {j + 1}" for j in range(dimensions)]
print()
print(pd.DataFrame(coordinates, index=labels, columns=dims).round(4))

## Rotated map

In [ ]:
# Turned clockwise by the angle on the first two dimensions:
# x' = x cos(a) + y sin(a), y' = -x sin(a) + y cos(a). Distances do not change.
angle = 0
rotated = coordinates.copy()
if dimensions >= 2:
    a = np.deg2rad(angle)
    rotated[:, :2] = coordinates[:, :2] @ np.array([[np.cos(a), -np.sin(a)],
                                                    [np.sin(a), np.cos(a)]])
print(f"\nRotation angle: {angle} degrees")
print(pd.DataFrame(rotated, index=labels, columns=dims).round(4))

## What the result says

Metric multidimensional scaling (MDS) rebuilds positions from a table of distances alone. The input here is the dissimilarity, that is the distance, between each pair of seven cities.

The eigenvalues say how many dimensions are needed. Mardia's first criterion reaches 0.9986 with two dimensions, so two dimensions rebuild the distances almost exactly, as a flat map should. The other eigenvalues are close to zero.

In the projection, dimension 2 puts Amsterdam (2.04) and Berlin (2.03) at the top and Zurich (-3.66) at the bottom, so north and south are in the right places. But dimension 1 puts London at 4.14 and Berlin at -5.16: west is on the right and east on the left. Distances cannot tell left from right. Flip that axis and the real map appears.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.